# Step 4 — Evaluate RAG Quality
Builds a 20-question ground-truth eval set, then measures:
- **Retrieval Precision** — are the retrieved chunks relevant?
- **Answer Faithfulness** — does the answer stay grounded in context?

**Constraints:** LLM-as-judge runs entirely via SNOWFLAKE.CORTEX.COMPLETE — no external APIs.

In [ ]:
import json
import pandas as pd
from snowflake.snowpark import Session
from snowflake.core import Root

try:
    session = Session.builder.getOrCreate()
except Exception:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()

TARGET_DB     = "RAG_HACKATHON_DB"
TARGET_SCHEMA = "RAG_SCHEMA"
TARGET_WH     = "RAG_SEARCH_WH"
SERVICE_NAME  = "ECONOMIC_SEARCH"
JUDGE_MODEL   = "llama3.1-70b"
GEN_MODEL     = "llama3.1-70b"
TOP_K         = 5

session.sql(f"USE DATABASE {TARGET_DB}").collect()
session.sql(f"USE SCHEMA {TARGET_SCHEMA}").collect()
session.sql(f"USE WAREHOUSE {TARGET_WH}").collect()

root = Root(session)
svc = root.databases[TARGET_DB].schemas[TARGET_SCHEMA].cortex_search_services[SERVICE_NAME]

def llm_complete(model, prompt):
    p = prompt.replace("'", "''")
    r = session.sql(f"SELECT SNOWFLAKE.CORTEX.COMPLETE('{model}', '{p}') AS resp").collect()
    return r[0]['RESP']

print("Eval session ready")
print(f"Judge: {JUDGE_MODEL}  |  Gen: {GEN_MODEL}  |  TOP_K: {TOP_K}")

In [ ]:
EVAL_SET = [
    # Banking — FDIC institutions (5)
    {'q': 'What information does the FDIC dataset contain about banking institutions?', 'type': 'factual', 'source': 'banking'},
    {'q': 'What bank classifications exist in the FDIC data?', 'type': 'factual', 'source': 'banking'},
    {'q': 'What geographic fields are available for banking institutions?', 'type': 'factual', 'source': 'banking'},
    {'q': 'What data is available about failed banks in the FDIC dataset?', 'type': 'factual', 'source': 'banking'},
    {'q': 'Summarize the types of banking institutions tracked by the FDIC.', 'type': 'summarization', 'source': 'banking'},
    # Banking — FRED economic indicators (5)
    {'q': 'What unemployment metrics are tracked in the FRED data?', 'type': 'factual', 'source': 'banking'},
    {'q': 'What interest rate and Treasury yield data is available?', 'type': 'factual', 'source': 'banking'},
    {'q': 'What labor market indicators are in the FRED financial labor dataset?', 'type': 'factual', 'source': 'banking'},
    {'q': 'What consumer price index data is available in the dataset?', 'type': 'factual', 'source': 'banking'},
    {'q': 'Summarize the macroeconomic indicators tracked in the FRED data.', 'type': 'summarization', 'source': 'banking'},
    # Demographics (7)
    {'q': 'What population breakdown variables exist in the demographics data?', 'type': 'factual', 'source': 'demographics'},
    {'q': 'What poverty status categories are tracked in the demographics dataset?', 'type': 'factual', 'source': 'demographics'},
    {'q': 'What education levels are tracked in the demographics data?', 'type': 'factual', 'source': 'demographics'},
    {'q': 'What housing characteristics are available in the demographics data?', 'type': 'factual', 'source': 'demographics'},
    {'q': 'Summarize the employment and income data in the economic characteristics.', 'type': 'summarization', 'source': 'demographics'},
    {'q': 'What household composition data exists in the demographics dataset?', 'type': 'factual', 'source': 'demographics'},
    {'q': 'What social characteristics like marital status and veteran status are tracked?', 'type': 'factual', 'source': 'demographics'},
    # Cross-dataset synthesis (3)
    {'q': 'How do the banking and demographics datasets complement each other geographically?', 'type': 'synthesis', 'source': None},
    {'q': 'What economic indicators from FRED could relate to poverty statistics in demographics?', 'type': 'synthesis', 'source': None},
    {'q': 'What connections can be drawn between demographic employment data and FDIC institution data?', 'type': 'synthesis', 'source': None},
]
print(f'Eval set loaded: {len(EVAL_SET)} questions')
print(pd.DataFrame(EVAL_SET)[['q','type','source']].to_string(index=False))

In [ ]:
def retrieve(q, k=TOP_K, src=None):
    kwargs = dict(query=q, columns=['chunk_text','source','title','doc_id'], limit=k)
    if src:
        kwargs['filter'] = {'@eq': {'source': src}}
    return json.loads(svc.search(**kwargs).json())['results']

def judge_precision(question, chunks):
    relevant = 0
    for chunk in chunks:
        p = f"""You are evaluating a RAG system. Does this retrieved text contain ANY information that could help answer the question, even partially? A chunk about the same dataset or topic counts as relevant.
Reply ONLY 'yes' or 'no'.
Question: {question}
Text: {chunk['chunk_text'][:500]}"""
        r = llm_complete(JUDGE_MODEL, p).strip().lower()
        if 'yes' in r:
            relevant += 1
    return relevant / len(chunks) if chunks else 0

def judge_faithfulness(question, answer, chunks):
    ctx = '\n'.join([c['chunk_text'][:400] for c in chunks])
    p = f"""You are evaluating a RAG system. Rate how faithfully the ANSWER uses only information from the CONTEXT.
IMPORTANT RULES:
- An answer that correctly says 'the data does not cover this' is FULLY faithful (1.0)
- An answer that cites specific data from the context is faithful (0.8-1.0)
- Only mark as unfaithful (below 0.5) if the answer contains specific claims NOT found in the context
Return ONLY valid JSON: {{"score": <0.0-1.0>, "reason": "<one sentence>"}}

Context: {ctx[:2000]}
Question: {question}
Answer: {answer[:800]}"""
    raw = llm_complete(JUDGE_MODEL, p).strip()
    try:
        start = raw.find('{')
        end   = raw.rfind('}') + 1
        return json.loads(raw[start:end])
    except:
        return {'score': 0.5, 'reason': 'parse error'}

print('Judge functions ready')

In [ ]:
results = []

for i, item in enumerate(EVAL_SET):
    q   = item['q']
    src = item.get('source')

    print(f'[{i+1:02d}/{len(EVAL_SET)}] {q[:60]}...')

    chunks = retrieve(q, src=src)

    ctx = '\n'.join([
        f"[Source {j+1}: {c.get('title','N/A')}]\n{c['chunk_text']}"
        for j, c in enumerate(chunks)
    ])

    prompt = f"""You are an economic intelligence assistant.
Answer using ONLY the context provided below.
Include [Source N] citations after the facts you use.
If the context is insufficient, say: 'The available data does not cover this topic.'

Context:
{ctx}

Question: {q}

Answer:"""

    answer = llm_complete(GEN_MODEL, prompt)

    precision    = judge_precision(q, chunks)
    faithfulness = judge_faithfulness(q, answer, chunks)

    results.append({
        'question'         : q,
        'type'             : item['type'],
        'source'           : src or 'both',
        'answer'           : answer,
        'precision'        : precision,
        'faithfulness'     : faithfulness['score'],
        'faith_reason'     : faithfulness['reason'],
        'chunks_retrieved' : len(chunks)
    })

df_eval = pd.DataFrame(results)

print('\nEvaluation complete!')
print(f'Avg Precision   : {df_eval["precision"].mean():.3f}')
print(f'Avg Faithfulness: {df_eval["faithfulness"].mean():.3f}')
print('\nBy question type:')
print(df_eval.groupby('type')[['precision', 'faithfulness']].mean().round(3).to_string())

In [ ]:
# CELL 5 — Save eval results to Snowflake (final)

session.sql(f"""
CREATE OR REPLACE TABLE {TARGET_DB}.{TARGET_SCHEMA}.EVAL_RESULTS (
    question             VARCHAR,
    question_type        VARCHAR,
    data_source          VARCHAR,
    answer               VARCHAR,
    precision_score      FLOAT,
    faithfulness_score   FLOAT,
    faithfulness_reason  VARCHAR,
    chunks_retrieved     INT,
    evaluated_at         TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
""").collect()

for _, row in df_eval.iterrows():
    safe_q = str(row['question']).replace("'", "\\'")
    safe_t = str(row['type']).replace("'", "\\'")
    safe_s = str(row['source']).replace("'", "\\'")
    safe_a = str(row['answer']).replace("'", "\\'")
    safe_r = str(row['faith_reason']).replace("'", "\\'")
    safe_p = float(row['precision']) if pd.notna(row['precision']) else 0.0
    safe_f = float(row['faithfulness']) if pd.notna(row['faithfulness']) else 0.0
    safe_c = int(row['chunks_retrieved']) if pd.notna(row['chunks_retrieved']) else 0

    session.sql(f"""
    INSERT INTO {TARGET_DB}.{TARGET_SCHEMA}.EVAL_RESULTS
    VALUES (
        '{safe_q}',
        '{safe_t}',
        '{safe_s}',
        '{safe_a}',
        {safe_p},
        {safe_f},
        '{safe_r}',
        {safe_c},
        CURRENT_TIMESTAMP()
    )
    """).collect()

print('Eval results saved to EVAL_RESULTS table')

print('\n' + '=' * 50)
print('EVALUATION SUMMARY')
print('=' * 50)
print(f'Total questions : {len(df_eval)}')
print(f'Avg Precision   : {df_eval["precision"].mean():.1%}')
print(f'Avg Faithfulness: {df_eval["faithfulness"].mean():.1%}')
print(f'Questions > 0.8 faith: {(df_eval["faithfulness"] > 0.8).sum()}/{len(df_eval)}')
print('\nStep 4 COMPLETE')